In [1]:
!pip install -q tensorflow-model-optimization tf_keras

import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import time
import tempfile
import numpy as np
import tensorflow as tf
import tensorflow_model_optimization as tfmot

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.7/242.7 kB 5.4 MB/s eta 0:00:00


Load & Prepare CIFAR-10 (Subset Resized to 96×96)

In [2]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# 5,000 train images, 1,000 test images
x_train_sub = x_train[:5000]
y_train_sub = y_train[:5000]
x_test_sub = x_test[:1000]
y_test_sub = y_test[:1000]

def preprocess(images):
    # Resize to 96x96 and apply MobileNetV2 scaling ([-1, 1])
    images = tf.image.resize(images, (96, 96))
    return tf.keras.applications.mobilenet_v2.preprocess_input(images)

x_train_sub = preprocess(x_train_sub)
x_test_sub = preprocess(x_test_sub)

170498071/170498071 [==============================] - 35s 0us/step


Single Functional Model

In [3]:
inputs = tf.keras.Input(shape=(96, 96, 3))

# Instantiate base model directly connected to inputs
base_model = tf.keras.applications.MobileNetV2(
    input_tensor=inputs,
    alpha=0.35,
    include_top=False,
    weights='imagenet'
)

# Connect classification head functionally
x = base_model.output
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(10)(x)

baseline_model = tf.keras.Model(inputs=inputs, outputs=outputs)

baseline_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

# Fine-tune briefly
baseline_model.fit(
    x_train_sub, y_train_sub,
    batch_size=64,
    epochs=5,
    validation_split=0.1
)

2019640/2019640 [==============================] - 0s 0us/step
Epoch 1/5
71/71 [==============================] - 76s 854ms/step - loss: 1.2393 - accuracy: 0.5949 - val_loss: 3.2447 - val_accuracy: 0.4500
Epoch 2/5
71/71 [==============================] - 58s 819ms/step - loss: 0.5396 - accuracy: 0.8162 - val_loss: 4.3564 - val_accuracy: 0.4320
Epoch 3/5
71/71 [==============================] - 58s 813ms/step - loss: 0.3458 - accuracy: 0.8807 - val_loss: 5.5677 - val_accuracy: 0.3640
Epoch 4/5
71/71 [==============================] - 59s 829ms/step - loss: 0.2623 - accuracy: 0.9118 - val_loss: 4.9995 - val_accuracy: 0.4380
Epoch 5/5
71/71 [==============================] - 57s 809ms/step - loss: 0.2395 - accuracy: 0.9198 - val_loss: 3.9362 - val_accuracy: 0.4600


Post-Training Quantization (Full INT8)

In [4]:
def representative_data_gen():
    for i in range(100):
        # Pass batches of 1 in float32 for calibration
        yield [tf.expand_dims(x_train_sub[i], axis=0)]

# Convert Float32 baseline
conv_float = tf.lite.TFLiteConverter.from_keras_model(baseline_model)
float32_tflite = conv_float.convert()

# Convert Full INT8 PTQ
conv_ptq = tf.lite.TFLiteConverter.from_keras_model(baseline_model)
conv_ptq.optimizations = [tf.lite.Optimize.DEFAULT]
conv_ptq.representative_dataset = representative_data_gen
conv_ptq.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv_ptq.inference_input_type = tf.int8
conv_ptq.inference_output_type = tf.int8
ptq_int8_tflite = conv_ptq.convert()

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Quantization-Aware Training (Full INT8)

In [5]:
# Insert FakeQuant nodes across the functional graph
q_aware_model = tfmot.quantization.keras.quantize_model(baseline_model)

q_aware_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

# Fine-tune with QAT on a small subset
q_aware_model.fit(
    x_train_sub[:1000], y_train_sub[:1000],
    batch_size=32,
    epochs=2,
    validation_split=0.1
)

# Convert Full INT8 QAT
conv_qat = tf.lite.TFLiteConverter.from_keras_model(q_aware_model)
conv_qat.optimizations = [tf.lite.Optimize.DEFAULT]
conv_qat.inference_input_type = tf.int8
conv_qat.inference_output_type = tf.int8
qat_int8_tflite = conv_qat.convert()

Epoch 1/2
29/29 [==============================] - 39s 590ms/step - loss: 1.6482 - accuracy: 0.5744 - val_loss: 7.3682 - val_accuracy: 0.1500
Epoch 2/2
29/29 [==============================] - 14s 496ms/step - loss: 1.1034 - accuracy: 0.6444 - val_loss: 7.4058 - val_accuracy: 0.1400


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Benchmarking

In [9]:
def benchmark_model(tflite_model, test_data, test_labels, is_int8=False):
    interpreter = tf.lite.Interpreter(model_content=tflite_model)
    interpreter.allocate_tensors()

    in_det = interpreter.get_input_details()[0]
    out_det = interpreter.get_output_details()[0]
    scale, zero_pt = in_det.get("quantization", (0.0, 0))

    correct = 0
    total_time_ms = 0.0

    for img, lbl in zip(test_data, test_labels):
        x = np.expand_dims(img, axis=0)
        if is_int8 and scale > 0:
            x = np.clip(np.round(x / scale + zero_pt), -128, 127).astype(np.int8)
        else:
            x = x.astype(np.float32)

        interpreter.set_tensor(in_det['index'], x)

        # Profile invoke latency strictly
        t0 = time.perf_counter()
        interpreter.invoke()
        total_time_ms += (time.perf_counter() - t0) * 1000.0

        # get_tensor() makes an independent copy, releasing the internal buffer reference
        output = interpreter.get_tensor(out_det['index'])
        pred = np.argmax(output[0])

        # Safely extract ground-truth label regardless of (N, 1) or (N,) shape
        true_label = lbl[0] if np.ndim(lbl) > 0 else lbl
        if pred == true_label:
            correct += 1

    # Measure file size in KB
    with tempfile.NamedTemporaryFile(suffix='.tflite', delete=False) as f:
        f.write(tflite_model)
        temp_path = f.name
    size_kb = os.path.getsize(temp_path) / 1024.0
    os.remove(temp_path)

    acc = (correct / len(test_data)) * 100.0
    avg_lat = total_time_ms / len(test_data)
    return size_kb, acc, avg_lat

# Run evaluation and fill the comparison table
res_f32 = benchmark_model(float32_tflite, x_test_sub, y_test_sub, is_int8=False)
res_ptq = benchmark_model(ptq_int8_tflite, x_test_sub, y_test_sub, is_int8=True)
res_qat = benchmark_model(qat_int8_tflite, x_test_sub, y_test_sub, is_int8=True)

print(f"\n{'Model':<18} | {'Size (KB)':<10} | {'Test Acc (%)':<12} | {'Avg Latency (ms)':<16}")
print("-" * 66)
print(f"{'Float32 baseline':<18} | {res_f32[0]:<10.1f} | {res_f32[1]:<12.2f} | {res_f32[2]:<16.2f}")
print(f"{'PTQ full INT8':<18} | {res_ptq[0]:<10.1f} | {res_ptq[1]:<12.2f} | {res_ptq[2]:<16.2f}")
print(f"{'QAT full INT8':<18} | {res_qat[0]:<10.1f} | {res_qat[1]:<12.2f} | {res_qat[2]:<16.2f}")

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



Model              | Size (KB)  | Test Acc (%) | Avg Latency (ms)
------------------------------------------------------------------
Float32 baseline   | 1594.6     | 48.00        | 2.24            
PTQ full INT8      | 607.7      | 46.10        | 2.10            
QAT full INT8      | 613.3      | 9.60         | 1.80            
